<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>VAE on dSprites</h1>

</div>

---

Training a variational autoencoder on the [dSprites](https://github.com/google-deepmind/dsprites-dataset) dataset using [Ion](https://github.com/auxeno/ion).

In [ ]:
# !pip install git+https://github.com/auxeno/ion optax plotly

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## The dataset

[dSprites](https://github.com/google-deepmind/dsprites-dataset) is a synthetic dataset of 2D shapes (heart, square, ellipse) procedurally generated from 5 independent ground truth factors:

| Factor | Values |
|---|---|
| Shape | 3 (square, ellipse, heart) |
| Scale | 6 levels |
| Orientation | 40 angles |
| X position | 32 steps |
| Y position | 32 steps |

All combinations are present, giving 737,280 binary 64x64 images. Because the factors are known and independent, dSprites is a standard benchmark for evaluating disentangled representation learning.

In [ ]:
from examples._common.datasets import load_dsprites

images = load_dsprites()
print(f"Shape: {images.shape}")
print(f"Dtype: {images.dtype}")

In [ ]:
# Plot a grid of random samples
from plotly.subplots import make_subplots

num_rows, num_cols = 2, 6
img_size = 128

rng = np.random.default_rng(0)
sample_indices = rng.choice(len(images), size=num_rows * num_cols, replace=False)
samples = images[sample_indices, :, :, 0]

fig = make_subplots(rows=num_rows, cols=num_cols, horizontal_spacing=0.02, vertical_spacing=0.02)
for i, img in enumerate(samples):
    row, col = divmod(i, num_cols)
    fig.add_trace(go.Image(z=np.stack([img * 255] * 3, axis=-1)), row=row + 1, col=col + 1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(
    height=num_rows * img_size,
    width=num_cols * img_size,
    title="dSprites samples",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

## Building the VAE

A VAE learns a compressed latent representation by training an encoder and decoder jointly. The encoder maps an image to a distribution over latent vectors (parameterized by a mean and log-variance), the decoder reconstructs images from samples of that distribution. The loss combines a reconstruction term (binary cross-entropy, since dSprites images are binary) with a KL divergence term that regularizes the latent space toward a standard normal.

We set `latent_dim=5` to match the 5 ground truth factors in dSprites. If the model learns a disentangled representation, each latent dimension should correspond to one of the underlying factors (shape, scale, orientation, x, y).

In [ ]:
from jaxtyping import Array, Float, PRNGKeyArray

class Encoder(nn.Module):
    conv_1: nn.Conv
    conv_2: nn.Conv
    conv_3: nn.Conv
    dense: nn.Linear
    dense_mean: nn.Linear
    dense_logvar: nn.Linear

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key, 6)
        self.conv_1 = nn.Conv(1, 16, kernel_shape=(4, 4), stride=2, padding=1, key=keys[0])
        self.conv_2 = nn.Conv(16, 32, kernel_shape=(4, 4), stride=2, padding=1, key=keys[1])
        self.conv_3 = nn.Conv(32, 64, kernel_shape=(4, 4), stride=2, padding=1, key=keys[2])
        self.dense = nn.Linear(4096, 64, key=keys[3])
        self.dense_mean = nn.Linear(64, 5, key=keys[4])
        self.dense_logvar = nn.Linear(64, 5, key=keys[5])

    def __call__(
        self, 
        x: Float[Array, "... 64 64 1"]
    ) -> tuple[Float[Array, "... 5"], Float[Array, "... 5"]]:
        x = jax.nn.relu(self.conv_1(x))
        x = jax.nn.relu(self.conv_2(x))
        x = jax.nn.relu(self.conv_3(x))

        x = x.reshape(*x.shape[:-3], -1)

        x = jax.nn.relu(self.dense(x))
        mean = self.dense_mean(x)
        logvar = self.dense_logvar(x)

        return mean, logvar
